In [14]:
"""
End-to-end test for tracer.ask() to verify placeholder fix.
"""
import sys
from pathlib import Path

# Ensure src is in path
# sys.path.insert(0, str(Path(__file__).parent / "src"))

from pylogtracer import LogTracer

# Create sample log file with MySQL errors and INC1000004
sample_log = """
2026-04-01 08:15:30 ERROR: INC1000004 - MySQL connection failed: [Errno 111] Connection refused
2026-04-01 08:16:45 INFO: INC1000004 - Retrying MySQL connection...
2026-04-01 08:17:12 ERROR: INC1000004 - MySQL query timeout: SELECT * FROM users timed out after 30s
2026-04-01 08:18:00 CRITICAL: INC1000004 - Failed to establish connection after 3 retries
2026-04-01 08:18:30 INFO: INC1000004 - Root cause identified: Database server was down for maintenance
2026-04-01 08:20:00 INFO: INC1000004 - Incident resolved - MySQL service restored
2026-04-01 08:20:30 INFO: Application started
2026-04-01 09:05:22 ERROR: INC1000005 - MySQL syntax error: SELCT * FROM orders (typo in query)
2026-04-01 09:10:15 WARNING: INC1000005 - Slow MySQL query detected: 5.2s for report_generation
2026-04-01 10:30:45 ERROR: INC1000006 - MySQL access denied: User 'app'@'localhost' has no permission
2026-04-01 11:00:00 INFO: Database maintenance started
2026-04-01 11:05:10 ERROR: INC1000007 - MySQL deadlock detected: INSERT/UPDATE conflict on users table
2026-04-01 12:30:00 INFO: Task completed successfully
2026-04-01 13:45:22 ERROR: INC1000008 - Connection pool exhausted: No available MySQL connections
2026-04-01 13:45:22 INFO:  incident no INC1000004 prediction is {"problem_context":"1234567890","problem_behavior":"abc"}
2026-04-01 13:46:22 INFO: prediction completed for INC1000004 
2026-04-01 13:45:22 INFO: incident no INC1000004 prediction is {"problem_context":"1234567890","problem_behavior":"abc"}

"""

# Write sample log
log_file = Path("application_logs.txt")
log_file.write_text(sample_log)
print(f"[INFO] Created sample log file: {log_file}")


[INFO] Created sample log file: application_logs.txt


In [20]:
# Debug: Check where pylogtracer is loaded from
import sys
import pylogtracer
print(f"PyLogTracer location: {pylogtracer.__file__}")
print(f"Smart reader location: {pylogtracer.preprocessing.smart_reader.__file__}")

# Now reinitialize tracer
from pylogtracer import LogTracer

tracer = LogTracer(
    file_path="application_logs.txt",
    llm_config={
        "provider": "ollama",
        # "model": "qwen2.5:3b",
        "model": "qwen3:latest",
        "base_url": "http://localhost:11434"
    }
)
print("✓ Tracer reinitialized")


PyLogTracer location: C:\D_drive\llm_research_work\pylogtracer\src\pylogtracer\__init__.py
Smart reader location: C:\D_drive\llm_research_work\pylogtracer\src\pylogtracer\preprocessing\smart_reader.py
✓ Tracer reinitialized


In [3]:
# Test: Direct call to search() to see the print statement
print("=== CALLING SEARCH ===")
result = tracer.search("INC1000004")
print("\n=== SEARCH RESULT ===")
print(f"Found {result['total_found']} entries")
print(f"First entry: {result['entries'][0] if result['entries'] else 'None'}")


[SEARCH TRACE] keyword=INC1000004, max_results=20
[SEARCH TRACE] found 9 entries


=== CALLING SEARCH ===
[SEARCH] Searching logs for keyword 'INC1000004' with max_results=20...
----------------------- ['2026-04-01 08:15:30 ERROR: INC1000004 - MySQL connection failed: [Errno 111] Connection refused', '2026-04-01 08:16:45 INFO: INC1000004 - Retrying MySQL connection...', '2026-04-01 08:17:12 ERROR: INC1000004 - MySQL query timeout: SELECT * FROM users timed out after 30s', '2026-04-01 08:18:00 CRITICAL: INC1000004 - Failed to establish connection after 3 retries', '2026-04-01 08:18:30 INFO: INC1000004 - Root cause identified: Database server was down for maintenance', '2026-04-01 08:20:00 INFO: INC1000004 - Incident resolved - MySQL service restored', '2026-04-01 08:20:30 INFO: Application started', '2026-04-01 09:05:22 ERROR: INC1000005 - MySQL syntax error: SELCT * FROM orders (typo in query)', '2026-04-01 09:10:15 WARNING: INC1000005 - Slow MySQL query detected: 5.2s for report_generation', "2026-04-01 10:30:45 ERROR: INC1000006 - MySQL access denied: User 'app'@'l

In [16]:
answer = tracer.ask("what is the prediction result of INC1000004? and list out all the  logs related to this incident")


  [QAAgent] Splitting question...
  [LLMFactory] provider=ollama | model=qwen2.5:3b
  [QAAgent] Splitter raw  : '[\n    {"id": 0, "question": "Search for the prediction result of INC1000004 in the logs.", "depends_on": null},\n    {"id": 1, "question": "Search for ALL log entries (including INFO, DEBUG, WARNING, ERROR) that contain INC1000004.", "depends_on": null}\n}'
  [QAAgent] Splitter clean: '[\n    {"id": 0, "question": "Search for the prediction result of INC1000004 in the logs.", "depends_on": null},\n    {"id": 1, "question": "Search for ALL log entries (including INFO, DEBUG, WARNING, ERROR) that contain INC1000004.", "depends_on": null}\n]'
  [QAAgent] Split into 2 sub-question(s):
    Q0: Search for the prediction result of INC1000004 in the logs.
    Q1: Search for ALL log entries (including INFO, DEBUG, WARNING, ERROR) that contain INC1000004.
  [QAAgent] Resolving time for current sub-question...
  [QAAgent] Thinking...


[SEARCH TRACE] keyword=INC1000004, max_results=20
[SEARCH TRACE] found 9 entries


  [QAAgent] Q0 step 1: TOOL: search ARGS: {"keyword": "INC1000004"} REASON: To find any log entries containing the specific...
[SystemMessage(content='You are a powerful log analysis agent.\nYou have access to tools to analyze log files. Use them to answer the user\'s question completely.\n\nTOOLS AVAILABLE:\n  error_frequency(date?, from_dt?, to_dt?)     — count errors by type\n  errors_by_date(date)                         — all errors on a specific date\n  errors_in_range(from_dt, to_dt)              — errors between two timestamps\n  last_incident()                              — most recent error cluster\n  summary(date?, from_dt?, to_dt?)             — high-level log overview\n  root_cause(date?, from_dt?, to_dt?)          — LLM root cause analysis\n  health_check()                               — is the system healthy?\n  incident_duration(date?, from_dt?, to_dt?)   — how long did incident last?\n  search(keyword)                              — search logs for any string or ID\n

[SEARCH TRACE] keyword=INC1000004, max_results=20
[SEARCH TRACE] found 9 entries


  [QAAgent] Q1 step 1: TOOL: search ARGS: {"keyword": "INC1000004"} REASON: To find all log entries containing the keyword...
[SystemMessage(content='You are a powerful log analysis agent.\nYou have access to tools to analyze log files. Use them to answer the user\'s question completely.\n\nTOOLS AVAILABLE:\n  error_frequency(date?, from_dt?, to_dt?)     — count errors by type\n  errors_by_date(date)                         — all errors on a specific date\n  errors_in_range(from_dt, to_dt)              — errors between two timestamps\n  last_incident()                              — most recent error cluster\n  summary(date?, from_dt?, to_dt?)             — high-level log overview\n  root_cause(date?, from_dt?, to_dt?)          — LLM root cause analysis\n  health_check()                               — is the system healthy?\n  incident_duration(date?, from_dt?, to_dt?)   — how long did incident last?\n  search(keyword)                              — search logs for any string or ID\n 

In [17]:
print(answer)

The prediction result of INC1000004 is as follows: On April 1, 2026, at 13:45:22, the system logged that the prediction for INC1000004 was { "problem_context": "1234567890", "problem_behavior": "abc" }. At 13:46:22, it noted that the prediction for INC1000004 was completed. There were multiple instances of the prediction being logged with different contexts and behaviors on April 1, 2026, at 08:17:12 (error), 08:15:30 (error), and 08:18:00 (error). On April 1, 2026, at 08:18:30, the root cause of the issue was identified as "Database server was down for maintenance." Additionally, there were two error entries on April 1, 2026, at 08:15:30 and 08:17:12, indicating that attempts to establish a MySQL connection failed due to timeout or connection refused.

Here are all the log entries related to INC1000004 across different severity levels:

- INFO: incident no INC1000004 prediction is {"problem_context": "1234567890", "problem_behavior": "abc"} (2 times)
- INFO: prediction completed for I

In [21]:
answer = tracer.ask("list out all the errors while processing INC1000004?")

  [QAAgent] Splitting question...
  [LLMFactory] provider=ollama | model=qwen3:latest
  [QAAgent] Splitter raw  : '[\n  {"id": 0, "question": "Search for ALL log entries (including INFO, DEBUG, WARNING, ERROR) that contain INC1000004.", "depends_on": null}\n]'
  [QAAgent] Splitter clean: '[\n  {"id": 0, "question": "Search for ALL log entries (including INFO, DEBUG, WARNING, ERROR) that contain INC1000004.", "depends_on": null}\n]'
  [QAAgent] Split into 1 sub-question(s):
    Q0: Search for ALL log entries (including INFO, DEBUG, WARNING, ERROR) that contain INC1000004.
  [QAAgent] Resolving time for current sub-question...
  [QAAgent] Thinking...


[SEARCH TRACE] keyword=INC1000004, max_results=20
[SEARCH TRACE] found 9 entries


  [QAAgent] Q0 step 1: TOOL: search ARGS: {"keyword": "INC1000004"} REASON: The user wants all log entries containing the s...
[SystemMessage(content='You are a powerful log analysis agent.\nYou have access to tools to analyze log files. Use them to answer the user\'s question completely.\n\nTOOLS AVAILABLE:\n  error_frequency(date?, from_dt?, to_dt?)     — count errors by type\n  errors_by_date(date)                         — all errors on a specific date\n  errors_in_range(from_dt, to_dt)              — errors between two timestamps\n  last_incident()                              — most recent error cluster\n  summary(date?, from_dt?, to_dt?)             — high-level log overview\n  root_cause(date?, from_dt?, to_dt?)          — LLM root cause analysis\n  health_check()                               — is the system healthy?\n  incident_duration(date?, from_dt?, to_dt?)   — how long did incident last?\n  search(keyword)                              — search logs for any string or ID\n

In [22]:
print(answer)


Found 9 log entries containing **INC1000004**:  
1. **2026-04-01 08:15:30 ERROR**: MySQL connection failed: [Errno 111] Connection refused  
2. **2026-04-01 08:16:45 INFO**: Retrying MySQL connection...  
3. **2026-04-01 08:17:12 ERROR**: MySQL query timeout: SELECT * FROM users timed out after 30s  
4. **2026-04-01 08:18:00 CRITICAL**: Failed to establish connection after 3 retries  
5. **2026-04-01 08:18:30 INFO**: Root cause identified: Database server was down for maintenance  
6. **2026-04-01 08:20:00 INFO**: Incident resolved - MySQL service restored  
7. **2026-04-01 13:45:22 INFO**: Incident no INC1000004 prediction is {"problem_context":"1234567890","problem_behavior":"abc"}  
8. **2026-04-01 13:45:22 INFO**: (Duplicate) Incident no INC1000004 prediction is {"problem_context":"1234567890","problem_behavior":"abc"}  
9. **2026-04-01 13:46:22 INFO**: Prediction completed for INC1000004  

Key issues: MySQL connection failures, query timeouts, and resolved via database maintenanc